# Teaching and evaluation agents as parallel agents

In [1]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Setup and authentication complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Setup and authentication complete.


In [2]:
import json, time, re
from typing import Dict, Any

from google.genai import types
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.memory import InMemoryMemoryService
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search, AgentTool

print("✅ ADK imports ready.")


✅ ADK imports ready.


#  logging, retry config, memory/session

In [3]:
import os
import logging

# CLEAN PREVIOUS LOGS 
for log_file in ["logger.log", "web.log", "tunnel.log"]:
    if os.path.exists(log_file):
        os.remove(log_file)
        print(f"🧹 Cleaned up {log_file}")

# CONFIGURE LOGGING
logger = logging.getLogger("skillix")
if not logger.handlers:
    ch = logging.StreamHandler()
    ch.setFormatter(logging.Formatter("%(levelname)s:%(name)s: %(message)s"))
    logger.addHandler(ch)
logger.setLevel(logging.DEBUG)

# RETRY OPTIONS
retry_config = types.HttpRetryOptions(
    attempts=5,
    exp_base=7,
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504]
)

#  MEMORY & SESSION SERVICES 
memory_service = InMemoryMemoryService()
session_service = InMemorySessionService()

# SAFE MEMORY WRAPPERS 
def memory_write(key, value):
    try:
        memory_service.write(key, value)
    except Exception:
        try:
            memory_service.save(key, value)
        except Exception:
            globals()[key] = value

def memory_read(key):
    try:
        return memory_service.read(key)
    except Exception:
        try:
            return memory_service.get(key)
        except Exception:
            return globals().get(key)

print("✅ Logging, retry, memory, session ready (with log cleanup).")


✅ Logging, retry, memory, session ready (with log cleanup).


# sample context/syllabus and test messages

In [4]:
topic = "plant-diseases"

sample_context = {
    "topic": topic,
    "items": [
        {"title":"Fungal disease","text":"Fungal diseases in plants are caused by fungi; they often produce spores that spread by wind or water.","source":"trusted_article_1"},
        {"title":"Bacterial disease","text":"Bacterial infections cause spots and wilting; they spread via water splash and insects.","source":"trusted_article_2"}
    ],
    "summary":"Plant diseases include fungal and bacterial infections; prevention includes hygiene, resistant varieties, and crop rotation."
}

sample_syllabus = [
    {"question_id":"q1","text":"What is a fungal plant disease?","lesson":"Fungal plant diseases are caused by fungi and spread via spores.","reference":sample_context["items"][0]["text"], "rubric":{"type":"short_answer","max_score":100}},
    {"question_id":"q2","text":"How do bacterial plant diseases spread?","lesson":"Bacterial diseases spread via water splash and vectors like insects.","reference":sample_context["items"][1]["text"], "rubric":{"type":"short_answer","max_score":100}}
]

memory_write(f"topic::{topic}::context", sample_context)
memory_write(f"topic::{topic}::syllabus", sample_syllabus)

TEST_MESSAGES = [
    {"id":"m1","user":"u1","text":"?What causes leaf spots in plants?"},
    {"id":"m2","user":"u1","text":"Leaf spots are caused by fungal spores."},
    {"id":"m3","user":"u1","text":"?How to prevent fungal diseases?"},
    {"id":"m4","user":"u1","text":"Crop rotation and resistant varieties."},
    {"id":"m5","user":"u1","text":"Which phone is best?"}
]

print("✅ Sample context + syllabus stored. TEST_MESSAGES ready.")


✅ Sample context + syllabus stored. TEST_MESSAGES ready.


# create LLM agents and wrap in AgentTool 

In [5]:
grader_agent = LlmAgent(model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config), name="GraderAgent")
teaching_agent = LlmAgent(model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config), name="TeachingAgent")

eval_agent_tool = None
teach_agent_tool = None
evaluator_agent = None

try:
    eval_agent_tool = AgentTool(grader_agent)
    print("✅ AgentTool created from grader_agent.")
except Exception as e:
    logger.debug("AgentTool(grader_agent) failed: %s", e)

try:
    teach_agent_tool = AgentTool(teaching_agent)
    print("✅ AgentTool created from teaching_agent.")
except Exception as e:
    logger.debug("AgentTool(teaching_agent) failed: %s", e)

# try creating evaluator_agent with tool list if ADK accepts it (optional)
try:
    evaluator_agent = LlmAgent(model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
                               name="EvaluatorAgent",
                               tools=[eval_agent_tool] if eval_agent_tool is not None else None)
    print("✅ evaluator_agent created (tools attached if supported).")
except Exception as e:
    logger.debug("Could not create evaluator_agent with tools: %s", e)

print("✅ Agents instantiated (no direct LLM invocation).")


✅ AgentTool created from grader_agent.
✅ AgentTool created from teaching_agent.
✅ evaluator_agent created (tools attached if supported).
✅ Agents instantiated (no direct LLM invocation).


# rule-based evaluator + eval caller
tokenization and short-answer scoring (rule-based fallback)

In [6]:
import re

STOPWORDS = set(["the","is","a","an","and","or","to","of","in","on","for","by","with","that","this","are","be","as","it","its"])

def tokenize(text):
    text = (text or "").lower()
    tokens = re.findall(r"[a-z0-9]+", text)
    return [t for t in tokens if t not in STOPWORDS]

def short_answer_score(reference, answer, max_score=100):
    ref_tokens = tokenize(reference)
    ans_tokens = tokenize(answer)
    if not ref_tokens or not ans_tokens:
        return 0, 0.0
    matches = sum(1 for t in ref_tokens if t in ans_tokens)
    overlap_ratio = matches / len(ref_tokens)
    # keyword boost
    boost = 0.0
    keywords = ["fung", "spore", "bacteria", "bacterial", "virus", "fungus", "infection", "symptom"]
    for kw in keywords:
        if any(kw in tok for tok in ans_tokens):
            boost += 0.15
    overlap_ratio = min(1.0, overlap_ratio + boost)
    score = int(round(overlap_ratio * max_score))
    return score, overlap_ratio

def rule_based_evaluate(question_text, student_answer, rubric):
    ans = (student_answer or "").strip()
    qtype = rubric.get("type")
    max_score = int(rubric.get("max_score", 100))
    if not ans:
        return {"scores":{"overall":0},"score":0,"feedback":{"overall":"No answer provided."},"suggestion":"Please try again.","metadata":{"method":"rule"}}
    if qtype == "mcq":
        correct = rubric.get("correct_answer","").strip().lower()
        score = 100 if ans.lower() == correct else 0
        return {"scores":{"correctness":score},"score":score,"feedback":{"correctness":"Correct." if score==100 else "Incorrect."},"suggestion":"" if score==100 else f"Correct answer: {correct}","metadata":{"method":"rule"}}
    if qtype == "numeric":
        try:
            expected = float(rubric.get("expected"))
            tol = float(rubric.get("tolerance",0.0))
            got = float(ans)
            diff = abs(got - expected)
            score = 100 if diff <= tol else max(0, int(100 - (diff - tol)*100))
            return {"scores":{"correctness":score},"score":score,"feedback":{"correctness":f"Numeric diff {diff:.4f}"},"suggestion":"" if diff<=tol else f"Expected approx {expected}","metadata":{"method":"rule"}}
        except:
            return None
    if qtype == "short_answer":
        ref = rubric.get("reference") or question_text or ""
        score, ratio = short_answer_score(ref, ans, max_score=max_score)
        feedback = {"correctness": f"Overlap ratio {ratio:.2f}"}
        suggestion = "" if score >= max_score*0.6 else "Revise core concepts and include keywords from reference."
        return {"scores":{"correctness":score},"score":score,"feedback":feedback,"suggestion":suggestion,"metadata":{"method":"rule_short_answer","overlap":ratio}}
    return None

# Safe eval caller that prefers rule-based then AgentTool/evaluator_agent
def eval_tool_call(payload: dict):
    rb = rule_based_evaluate(payload.get("question_text"), payload.get("student_answer"), payload.get("rubric",{}))
    if rb is not None:
        rb["metadata"].update({"invocation_ts": time.time()})
        return rb
    # try evaluator_agent.call_tool if available
    try:
        if evaluator_agent is not None and hasattr(evaluator_agent, "call_tool"):
            try:
                return evaluator_agent.call_tool(eval_agent_tool or "evaluator", payload)
            except Exception as e:
                logger.debug("evaluator_agent.call_tool failed: %s", e)
    except Exception as e:
        logger.debug("evaluator_agent call attempt error: %s", e)
    # try eval_agent_tool as callable
    try:
        if eval_agent_tool is not None and callable(eval_agent_tool):
            return eval_agent_tool(payload)
    except Exception as e:
        logger.debug("eval_agent_tool callable attempt failed: %s", e)
    # fallback record
    return {"scores":{"overall":0},"score":0,"feedback":{"overall":"LLM not available; rule-based couldn't decide."},"suggestion":"Enable LLM or review manually.","metadata":{"method":"fallback","invocation_ts": time.time()}}


# Teaching tool caller (with off-topic guard)
teaching tool caller — prefer AgentTool else context fallback


In [7]:
def simple_text_similarity(a: str, b: str):
    A = set(tokenize(a))
    B = set(tokenize(b))
    if not A or not B:
        return 0.0
    inter = len(A & B)
    union = len(A | B)
    return inter / union if union>0 else 0.0

def teaching_tool_call(payload: dict):
    # Try AgentTool or evaluator_agent.call_tool if available (safe attempts)
    try:
        if teach_agent_tool is not None:
            if callable(teach_agent_tool):
                try:
                    return teach_agent_tool(payload)
                except Exception as e:
                    logger.debug("teach_agent_tool(payload) failed: %s", e)
            # attempt evaluator_agent.call_tool with teach_agent_tool as a tool if available
            try:
                if evaluator_agent is not None and hasattr(evaluator_agent, "call_tool"):
                    return evaluator_agent.call_tool(teach_agent_tool, payload)
            except Exception:
                pass
    except Exception as e:
        logger.debug("teaching AgentTool attempts error: %s", e)

    # deterministic context-based fallback (always available)
    item = payload.get("syllabus_item", {})
    context_items = payload.get("context_items", [])
    user_turn = (payload.get("user_turn") or "").strip()
    combined_context = " ".join([c.get("text","") for c in context_items])[:4000]

    # off-topic guard inside teaching tool
    sim = simple_text_similarity(user_turn, combined_context)
    if sim < 0.05:
        return {
            "text": "This question seems outside the current learning topic. Please ask something related to this topic or say 'Change topic to <name>'.",
            "metadata": {"method":"teaching_offtopic_guard","similarity":sim}
        }

    # produce context-based reply
    if user_turn and (user_turn.endswith("?") or user_turn.startswith("?") or "explain" in user_turn.lower()):
        words = set(w.lower() for w in user_turn.split() if len(w) > 3)
        match = None
        for c in context_items:
            txt = c.get("text","").lower()
            if any(w in txt for w in words):
                match = c.get("text")
                break
        if not match:
            match = item.get("lesson", context_items[0]["text"] if context_items else "Here is a short note.")
        resp = f"{match} (based on topic context.)"
    else:
        resp = item.get("lesson", context_items[0]["text"] if context_items else "Short lesson unavailable.")
    return {"text": resp, "metadata":{"method":"fallback_context","similarity":sim}}


# Orchestrator dispatch
orchestrator — routes to teaching/evaluation or polite off-topic

In [8]:
def orchestrator_dispatch(message: dict, topic_name: str):
    text = (message.get("text") or "").strip()
    syllabus = memory_read(f"topic::{topic_name}::syllabus") or []
    ctx = memory_read(f"topic::{topic_name}::context") or {}
    context_items = ctx.get("items", []) if isinstance(ctx, dict) else []
    context_text = " ".join([c.get("text","") for c in context_items])[:4000]

    # off-topic detection
    sim = simple_text_similarity(text, context_text)
    if sim < 0.05:
        return {"route":"none", "output":{"text": f"This question looks outside the topic '{topic_name}'. I can only help with that topic right now. If you'd like to switch topics, say 'Change topic to <new topic>'.","metadata":{"method":"orchestrator_offtopic","similarity":sim}}}

    # heuristics
    def looks_like_question(t: str) -> bool:
        if not t: return False
        if t.strip().startswith("?") or "?" in t: return True
        if any(w in t.lower() for w in ["what","why","how","explain","difference","when","where"]): return True
        return False

    def looks_like_answer(t: str) -> bool:
        if not t: return False
        if t.strip().replace(".", "", 1).isdigit(): return True
        if ("?" not in t) and (t.lower().startswith("i ") or "think" in t.lower() or len(t.split()) < 12): return True
        return False

    if looks_like_question(text):
        payload = {"syllabus_item": syllabus[0] if syllabus else {}, "context_items": context_items, "user_turn": text.lstrip("?").strip()}
        out = teaching_tool_call(payload)
        return {"route":"teaching", "output": out}

    if looks_like_answer(text):
        q_item = syllabus[0] if syllabus else {"question_id":"unknown","text":"","reference":"","rubric":{"type":"short_answer","max_score":100}}
        payload = {"question_id": q_item.get("question_id"), "question_text": q_item.get("text"), "student_answer": text, "reference_text": q_item.get("reference",""), "rubric": q_item.get("rubric",{})}
        out = eval_tool_call(payload)
        return {"route":"evaluation", "output": out}

    payload = {"syllabus_item": syllabus[0] if syllabus else {}, "context_items": context_items, "user_turn": text}
    out = teaching_tool_call(payload)
    return {"route":"teaching", "output": out}


# Simulation & save results
run simulation on TEST_MESSAGES and save results

In [9]:
results = []
for m in TEST_MESSAGES:
    print("\n--- MESSAGE:", m["id"], m["text"])
    res = orchestrator_dispatch(m, topic)
    print("ROUTE:", res["route"])
    print("OUTPUT:", res["output"])
    results.append({"message": m, "route": res["route"], "output": res["output"]})

# save results
with open("/kaggle/working/parallel_agents_cleaned_results.json","w") as f:
    json.dump(results, f, indent=2, default=str)
print("\nSaved -> /kaggle/working/parallel_agents_cleaned_results.json")



--- MESSAGE: m1 ?What causes leaf spots in plants?
ROUTE: teaching
OUTPUT: {'text': 'Bacterial infections cause spots and wilting; they spread via water splash and insects. (based on topic context.)', 'metadata': {'method': 'fallback_context', 'similarity': 0.08695652173913043}}

--- MESSAGE: m2 Leaf spots are caused by fungal spores.
ROUTE: evaluation
OUTPUT: {'scores': {'correctness': 55}, 'score': 55, 'feedback': {'correctness': 'Overlap ratio 0.55'}, 'suggestion': 'Revise core concepts and include keywords from reference.', 'metadata': {'method': 'rule_short_answer', 'overlap': 0.55, 'invocation_ts': 1763631157.8457992}}

--- MESSAGE: m3 ?How to prevent fungal diseases?
ROUTE: teaching
OUTPUT: {'text': 'Fungal diseases in plants are caused by fungi; they often produce spores that spread by wind or water. (based on topic context.)', 'metadata': {'method': 'fallback_context', 'similarity': 0.09090909090909091}}

--- MESSAGE: m4 Crop rotation and resistant varieties.
ROUTE: none
OUTP